In [1]:
import lightkurve as lk
import glob
from tqdm import tqdm
import matplotlib.pyplot as plt
import requests
from bs4 import BeautifulSoup
import numpy as np

%matplotlib widget

In [2]:
def access_tpfs():
    """
    """

    test_case = []

    lightkurve_file_folder = '/Users/zgl12/'

    files = sorted(glob.glob(lightkurve_file_folder + '*.fits.gz'))

    for file in tqdm(files, desc='Reading TPFs'):
        tpf = lk.read(file)
        test_case.append(tpf)
        
    return test_case

In [ ]:
x = access_tpfs()

In [4]:
y = x[0].flux.value.shape

In [ ]:
plt.figure()
plt.imshow(x[0].flux.value[0], origin='lower')
plt.show()

In [12]:
url = "https://archive.stsci.edu/pub/k2/download_scripts/target-pixel-files/c16/" # URL to the directory containing the files

wget_strings = [] # List to store the wget commands

response = requests.get(url) # Send a GET request to the URL

if response.status_code == 200: # Check if the request was successful
    soup = BeautifulSoup(response.text, 'html.parser') # Parse the HTML content of the page

    links = soup.find_all('a', href=True) # Find all anchor tags with href attribute (which link to the files)

    for link in tqdm(links, desc='Links analysed'): # Extract and print the filenames
        filename = link['href']
        if filename.endswith('.txt'):
            new_response = requests.get(url + filename)
            new_soup = BeautifulSoup(new_response.text, 'html.parser') # Parse the HTML content of the page
            for line in new_soup.get_text().split('\n'):
                wget_strings.append(line)
            
else:
    print(f"Failed to retrieve the page. Status code: {response.status_code}")


Links analysed:  82%|████████▏ | 70/85 [01:31<00:22,  1.50s/it]/var/folders/6v/pg_c9mqn3gn45gn_zmvy9w9cvfsx9j/T/ipykernel_55315/61201121.py:16: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  new_soup = BeautifulSoup(new_response.text, 'html.parser') # Parse the HTML content of the page
Links analysed: 100%|██████████| 85/85 [01:52<00:00,  1.32s/it]


In [14]:
np.savetxt('k2_c16_downloads.txt', np.array(wget_strings, dtype=str), fmt='%s', delimiter='\n')

58696